# Case TechShop - E-commerce Analytics

Notebook principal do case. Cada questão segue o padrão `[MD explicação] -> [CODE] -> [MD análise]`, exceto Q6 (markdown-only).

# Bloco 1: Setup

Imports centralizados, carga do dataset e configuração de caminhos.

In [11]:
import pandas as pd
import numpy as np
import duckdb
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

RAW_PATH = Path('../data/raw/ecommerce_vendas.csv')
INTERIM_DIR = Path('../data/interim')
SQL_DIR = Path('../sql')

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

print('Bibliotecas carregadas!')

Bibliotecas carregadas!


# Bloco 2: Q1 — Diagnóstico de Qualidade

Mapeamento de problemas de qualidade do dataset antes de qualquer tratamento.

Checklist coberto: shape e tipos, nulos por coluna, missing disfarçado, duplicatas, domínios categóricos, resumo estatístico numérico, outliers e inconsistências cruzadas.

In [12]:
df = pd.read_csv(RAW_PATH)

# --- 1. Shape e tipos: verificar se o dataset tem as 11 colunas esperadas e se os tipos estão corretos ---
print("=== Shape ===")
print(df.shape)
print("\n=== Tipos ===")
print(df.dtypes)
print("\n=== Amostra ===")
display(df.head())

# --- 2. Nulos: identificar colunas com ausências que impactam volume, receita e análise geográfica ---
print("\n=== Nulos absolutos e % ===")
nulos = df.isnull().sum()
percentual_nulos = (nulos / len(df) * 100).round(2)
display(pd.DataFrame({'nulos': nulos, 'percentual_%': percentual_nulos})[nulos > 0])

# --- 3. Missing disfarçado: capturar strings vazias que o pandas não converte automaticamente em NaN ---
print("\n=== Missing disfarçado ===")
colunas_texto = df.select_dtypes(include='object').columns
strings_vazias = [(col, (df[col].fillna('').str.strip() == '').sum())
                  for col in colunas_texto
                  if (df[col].fillna('').str.strip() == '').sum() > 0]
print('\n'.join(f'  {col}: {n}' for col, n in strings_vazias) if strings_vazias else '  nenhum')

# --- 4. Duplicatas: confirmar se pedido_id é chave primária confiável antes de qualquer agregação ---
print("\n=== Duplicatas em pedido_id ===")
linhas_duplicadas = df[df.duplicated('pedido_id', keep=False)]
print(f"  Linhas duplicadas: {len(linhas_duplicadas)}  |  pedido_ids afetados: {linhas_duplicadas['pedido_id'].nunique()}")
display(linhas_duplicadas.sort_values('pedido_id'))

# --- 5. Domínios: detectar valores fora do padrão em status, categoria e uf que fragmentariam agrupamentos ---
print("\n=== Distribuição: status ===")
print(df['status'].value_counts(dropna=False).to_string())

print("\n=== Distribuição: categoria ===")
print(df['categoria'].value_counts(dropna=False).to_string())

print("\n=== UFs presentes ===")
UFS_VALIDAS = {
    'AC','AL','AP','AM','BA','CE','DF','ES','GO','MA','MT','MS','MG',
    'PA','PB','PR','PE','PI','RJ','RN','RS','RO','RR','SC','SP','SE','TO'
}
ufs_presentes = set(df['uf'].dropna().unique())
ufs_invalidas = ufs_presentes - UFS_VALIDAS
print(f"  Únicas (excl. nulos): {sorted(ufs_presentes)}")
print(f"  Fora do domínio: {ufs_invalidas if ufs_invalidas else 'nenhuma'}")

# --- 6. Estatísticas: obter range e distribuição das colunas numéricas como base para o diagnóstico de outliers ---
print("\n=== Resumo estatístico ===")
display(df[['qtd', 'valor_unit', 'desconto_%', 'avaliacao']].describe())

# --- 7a. Validação de domínio: checar violações de regras de negócio com limites fixos conhecidos ---
# Método: limiar fixo. Aplicável a colunas com domínio bem definido (qtd, desconto_%, avaliacao).
# Não detecta outliers estatísticos em valor_unit, que não tem limite superior de negócio definido.
print("\n=== Validação de domínio (regras de negócio) ===")
print(f"  qtd <= 0: {(df['qtd'] <= 0).sum()}")
print(f"  qtd não-inteira: {((df['qtd'].dropna() % 1) != 0).sum()}")
print(f"  valor_unit <= 0: {(df['valor_unit'] <= 0).sum()}")
print(f"  desconto_% < 0: {(df['desconto_%'] < 0).sum()}")
print(f"  desconto_% > 100: {(df['desconto_%'] > 100).sum()}")
print(f"  avaliacao fora de [1,5]: {((df['avaliacao'] < 1) | (df['avaliacao'] > 5)).sum()}")

# --- 7b. Outliers estatísticos em valor_unit: IQR (robusto a distribuições assimétricas) ---
# IQR é preferível ao z-score aqui porque a distribuição de valor_unit é assimétrica
# (mediana R$259 << média R$484), o que viola a premissa de normalidade do z-score.
# O método IQR define dois limites: inferior (Q1 - 1,5×IQR) e superior (Q3 + 1,5×IQR).
q1_valor = df['valor_unit'].quantile(0.25)
q3_valor = df['valor_unit'].quantile(0.75)
iqr_valor = q3_valor - q1_valor
limite_inferior_valor = q1_valor - 1.5 * iqr_valor
limite_superior_valor = q3_valor + 1.5 * iqr_valor

registros_abaixo_limite = df[df['valor_unit'] < limite_inferior_valor]
registros_acima_limite = df[df['valor_unit'] > limite_superior_valor]

print(f"\n=== Outliers estatísticos: valor_unit (método IQR) ===")
print(f"  Q1: R$ {q1_valor:.2f}  |  Q3: R$ {q3_valor:.2f}  |  IQR: R$ {iqr_valor:.2f}")
print(f"  Limite inferior (Q1 - 1,5 × IQR): R$ {limite_inferior_valor:.2f}")
print(f"  Limite superior (Q3 + 1,5 × IQR): R$ {limite_superior_valor:.2f}")
print(f"  Registros abaixo do limite inferior: {len(registros_abaixo_limite)}"
      f"  — limite negativo; nenhum preço pode assumir valor abaixo de R$ {limite_inferior_valor:.2f}")
print(f"  Registros acima do limite superior: {len(registros_acima_limite)}")
if not registros_acima_limite.empty:
    print(f"  Produtos envolvidos (acima do limite):")
    display(
        registros_acima_limite
        .groupby(['produto', 'categoria'])['valor_unit']
        .agg(pedidos='count', preco_unitario='first')
        .sort_values('preco_unitario', ascending=False)
        .reset_index()
    )

# --- 8. Cruzamento avaliacao x status: verificar se apenas pedidos entregues possuem avaliação preenchida ---
print("\n=== Avaliação por status ===")
avaliacao_por_status = df.groupby('status', observed=True).agg(
    total_pedidos=('pedido_id', 'count'),
    com_avaliacao=('avaliacao', 'count')
)
avaliacao_por_status['pct_avaliados'] = (
    avaliacao_por_status['com_avaliacao'] / avaliacao_por_status['total_pedidos'] * 100
).round(1)
display(avaliacao_por_status)

# --- 9. Datas: detectar formatos inconsistentes que o pandas converterá em NaT e excluirá da análise temporal ---
print("\n=== data_pedido ===")
df['data_pedido_parsed'] = pd.to_datetime(df['data_pedido'], errors='coerce')
datas_invalidas = df['data_pedido_parsed'].isna().sum()
print(f"  Não parseável (NaT): {datas_invalidas}")
print(f"  Range (datas válidas): {df['data_pedido_parsed'].min().date()} a {df['data_pedido_parsed'].max().date()}")
print("  Exemplos de formato inválido:")
display(df.loc[df['data_pedido_parsed'].isna(), ['pedido_id', 'data_pedido']].head(5))

=== Shape ===
(1205, 11)

=== Tipos ===
pedido_id        int64
data_pedido        str
cliente_id         str
uf                 str
produto            str
categoria          str
qtd            float64
valor_unit     float64
desconto_%     float64
status             str
avaliacao      float64
dtype: object

=== Amostra ===


,pedido_id,data_pedido,cliente_id,uf,produto,categoria,qtd,valor_unit,desconto_%,status,avaliacao
0,1102,2024-03-02,C191,MG,HD Externo 2TB,Armazenamento,2.0,349.0,0.0,entregue,4.0
1,1947,2024-04-14,C036,PA,Toner HP,Impressoras,1.0,129.0,0.0,entregue,5.0
2,1307,2024-12-07,C031,MG,Mouse Gamer,Periféricos,1.0,89.0,10.0,em_transito,NaN
3,1110,2024-08-07,C079,MG,Headset 7.1,Periféricos,1.0,349.0,20.0,entregue,5.0
4,2062,2024-02-15,C119,SP,Suporte Notebook,Acessórios,2.0,89.9,0.0,entregue,4.0



=== Nulos absolutos e % ===


,nulos,percentual_%
cliente_id,25,2.07
uf,15,1.24
qtd,12,1.00
desconto_%,30,2.49
avaliacao,241,20.00



=== Missing disfarçado ===
  cliente_id: 25
  uf: 15

=== Duplicatas em pedido_id ===
  Linhas duplicadas: 10  |  pedido_ids afetados: 5


...

,produto,categoria,pedidos,preco_unitario
0,"Monitor 24""",Monitores,1,8990.0
1,HD Externo 2TB,Armazenamento,1,3490.0
2,Headset 7.1,Periféricos,2,3490.0
3,"Monitor 32"" 4K",Monitores,56,2499.0
4,"Monitor 32"" 4K",monitores,1,2499.0
5,SSD 500GB,Armazenamento,1,1890.0
6,"Monitor 27"" 144Hz",Monitores,55,1499.0
7,"Monitor 27"" 144Hz",monitores,1,1499.0



=== Avaliação por status ===


,total_pedidos,com_avaliacao,pct_avaliados
status,,,
Devolvido,2,2,100.0
Entregue,8,8,100.0
cancelado,138,0,0.0
devolvido,82,82,100.0
em_transito,103,0,0.0
entregue,872,872,100.0



=== data_pedido ===
  Não parseável (NaT): 20
  Range (datas válidas): 2024-01-01 a 2024-12-31
  Exemplos de formato inválido:


,pedido_id,data_pedido
44,2146,11/02/2024
108,1197,24/02/2024
159,1209,09/07/2024
242,1877,15/11/2024
372,1107,01/09/2024


### Achados — Q1

O dataset tem **`1.205` linhas e `11` colunas** e é utilizável — mas não está pronto para análise. Foram identificados `11` problemas distintos em cinco categorias: campos em branco, pedidos duplicados, inconsistências de formato, valores inválidos e datas mal formadas. Nenhum exige descarte do dataset, mas todos produzem distorções silenciosas em cálculos de receita, volume e satisfação se não forem tratados antes de Q2.

---

#### Campos em branco

| Coluna | Ausências | % | Impacto downstream |
|---|---|---|---|
| `avaliacao` | `241` | `20,00%` | Todos em `cancelado` (`138`) e `em_transito` (`103`) — padrão esperado; análises de satisfação exigem filtro por `status` antes de calcular médias |
| `desconto_%` | `30` | `2,49%` | Esses pedidos ficam fora de qualquer cálculo de margem líquida |
| `cliente_id` | `25` | `2,07%` | Pedidos anônimos não entram em segmentação nem análise de comportamento por cliente |
| `uf` | `15` | `1,24%` | Excluídos automaticamente de relatórios regionais; se concentrados em uma região, a análise geográfica estará sistematicamente distorcida |
| `qtd` | `12` | `1,00%` | Não contribuem para volume nem receita |

A verificação de missing disfarçado (strings vazias) retornou os mesmos `25` e `15` já capturados como `NaN` — confirma que não há ausências ocultas além dos nulos acima.

**Padrão de `avaliacao` por status:** `cancelado` (`138` pedidos) e `em_transito` (`103`) têm `0%` de avaliação; `entregue` (`872`) e `devolvido` (`82`) têm `100%`. O padrão é coerente — clientes avaliam ao receber. Ressalva importante: os `82` pedidos devolvidos têm nota preenchida, mas ela foi registrada antes da decisão de devolução. Incluir devolvidos em métricas de NPS ou satisfação infla o indicador, pois o dado não reflete a experiência pós-venda.

---

#### Pedidos duplicados

`10` linhas correspondem a apenas `5` pedidos distintos. Os pares diferem em natureza — e isso determina a estratégia de tratamento em Q2:

- **Cópias byte-a-byte idênticas** — pedidos `1135`, `1618` e `1906`: todas as colunas são iguais nas duas linhas. Qualquer cópia pode ser descartada diretamente.
- **Duplicatas com agravante** — pedido `1113`: uma cópia tem `cliente_id = NaN`, a outra tem `C019`; é necessário decidir qual preservar. Pedido `1885`: uma cópia tem `categoria = "Monitores"`, a outra `"monitores"` — carrega tanto duplicata quanto inconsistência de capitalização.

Sem remoção, qualquer contagem de pedidos ou somatório de receita estará inflado.

---

#### Inconsistências de formato

**`status`** — `10` registros grafados com inicial maiúscula (`Entregue`: `8`, `Devolvido`: `2`) contra o padrão minúsculo do restante. Um filtro por `"entregue"` ignora silenciosamente esses `10` casos, subestimando receita e volumes entregues.

O output revela também que `cancelado` representa `138` pedidos, equivalente a `~11,5%` do total — o segundo maior grupo depois de `entregue`. Esse volume de cancelamentos é um sinal operacional independente dos problemas de qualidade: qualquer análise de receita que não segmente por `status` estará misturando vendas efetivas com pedidos não concluídos.

**`categoria`** — `15` registros em lowercase distribuídos em seis categorias (`periféricos`: `6`, `armazenamento`: `3`, `acessórios`: `2`, `monitores`: `2`, `câmeras`: `1`, `impressoras`: `1`). Um `groupby` sem normalização produzirá `12` grupos distintos em vez dos `6` esperados, distorcendo participações de categoria e rankings de receita.

**`uf`** — todas as `13` UFs presentes são válidas. A cobertura de apenas `13` dos `27` estados é uma limitação do dataset, não um erro de qualidade.

---

#### Valores inválidos e outliers estatísticos

**Validação de domínio (regras de negócio fixas):**
- `qtd <= 0`: **`5` registros** (mínimo = `-1`) — quantidade negativa não tem interpretação válida; distorce volume e receita
- `qtd` não-inteira: `0` — confirma problema de tipo de armazenamento (`float64`), sem frações reais (DI-011)
- `valor_unit <= 0`: `0` | `desconto_%` fora de `[0, 100]`: `0` | `avaliacao` fora de `[1, 5]`: `0`

O `desconto_%` tem máximo observado de `20%`. O teto aparente pode refletir uma política de negócio; em Q2, qualquer registro com desconto acima de `20%` deve ser tratado como erro de entrada.

**Outliers estatísticos em `valor_unit` (método IQR):**

`valor_unit` não tem teto definido por regra de negócio, e sua distribuição é assimétrica (mediana `R$ 259` << média `R$ 484`, com desvio padrão `R$ 652,84` — maior que a própria média), o que invalida o z-score. O IQR é robusto a assimetrias e define dois limites:

- Limite inferior: `R$ 129 - 1,5 × R$ 470 = -R$ 576` — negativo; nenhum preço pode assumir esse valor → `0` registros abaixo (confirmado pela validação de domínio: `valor_unit <= 0 = 0`)
- Limite superior: `R$ 599 + 1,5 × R$ 470 = R$ 1.304`
- **`118` registros acima do limite**, correspondendo a `6` produtos distintos: Monitor 24" (`R$ 8.990`), Headset 7.1 e HD Externo 2TB (`R$ 3.490`), Monitor 32" 4K (`R$ 2.499`), SSD 500GB (`R$ 1.890`) e Monitor 27" 144Hz (`R$ 1.499`)

> A tabela do output exibe `8` linhas — não `6` — porque Monitor 32" 4K e Monitor 27" 144Hz aparecem duas vezes cada: uma com `categoria = "Monitores"` e outra com `"monitores"` (DI-009). O `groupby(['produto', 'categoria'])` trata os dois valores de capitalização como grupos distintos.

Esses `118` registros são produtos legítimos de alto valor — não erros de entrada. O achado confirma que o catálogo da TechShop tem cauda longa de preço: poucas SKUs de alta performance (monitores, armazenamento) concentram uma parcela desproporcional da receita, tornando a média por pedido um indicador pouco representativo da venda típica.

---

#### Datas mal formadas

`20` registros usam o formato `dd/mm/yyyy` (ex.: pedido `2146` → `"11/02/2024"`) em vez do padrão `yyyy-mm-dd`. O `pd.to_datetime` com `errors='coerce'` descarta silenciosamente esses `20` como `NaT`, excluindo-os de qualquer análise temporal — relatórios mensais, sazonalidade, tendências. O range das datas válidas cobre o ano inteiro: `2024-01-01` a `2024-12-31`.

---

#### O que isso significa para o negócio

Usar o dataset sem tratamento produz distorções concretas em pelo menos quatro frentes:

1. **Receita total superestimada** — os `5` pedidos duplicados entram duas vezes em qualquer soma de faturamento
2. **Relatórios regionais incompletos** — `15` pedidos sem `uf` somem de qualquer mapa ou ranking por estado; se concentrados em uma região, a análise geográfica estará sistematicamente subestimada
3. **Satisfação artificialmente inflada** — os `82` devolvidos têm nota registrada antes da devolução; incluí-los em médias de NPS eleva o indicador sem refletir a experiência pós-venda
4. **Agrupamentos quebrados sem aviso** — inconsistências de capitalização em `status` e `categoria` criam grupos duplicados em relatórios, tornando participações de mercado e rankings incorretos de forma silenciosa

---

#### Ressalvas

- O dataset cobre `13` dos `27` estados. Não é erro de qualidade, mas limita conclusões sobre cobertura geográfica da TechShop.
- Os `118` registros acima do limite IQR são produtos legítimos — não devem ser removidos, mas precisam ser considerados ao interpretar médias de receita.
- A ausência de avaliação em `cancelado` e `em_transito` é esperada e não representa dado problemático, desde que análises de satisfação filtrem por `status` antes de calcular médias.
- O diagnóstico cobre o dataset como está. Não é possível afirmar se os problemas vêm do sistema de origem ou de erros de exportação.

---

#### Prioridade para Q2

Os `11` achados foram catalogados em `memory-bank/data-issues.md` (DI-001 a DI-011). Ordem de tratamento recomendada:

1. **DI-002** — remover duplicatas de `pedido_id`; para `1113`, preservar a cópia com `cliente_id` preenchido
2. **DI-003** — excluir ou corrigir `qtd <= 0`
3. **DI-008 / DI-009** — normalizar capitalização de `status` e `categoria` (pré-requisito para qualquer agrupamento)
4. **DI-010** — reprocessar `data_pedido` suportando ambos os formatos
5. **DI-001 / DI-006 / DI-007** — decidir estratégia para campos com ausências (excluir, manter com marcação ou imputar, conforme o que cada análise exigir)

# Bloco 3: Q2 - Tratamento

## Q2 - Tratamento e Justificativa

_A preencher via `/tratar`._

# Bloco 4: Q3 - SQL

## Q3 - Consultas SQL

_A preencher via `/consultar-sql`._

# Bloco 5: Q4 - Negócio

## Q4 - Analise de Negocio Aberta

_A preencher via `/analisar-negocio`._

# Bloco 6: Q5 - Debug

## Q5 - Encontre o Erro

_A preencher via `/encontrar-erro`._

# Bloco 7: Q6 - Modelagem Dimensional

## Q6 - Modelagem

_A preencher via `/modelar-dimensional` (markdown-only)._

# Bloco 8: Q7 - Insight

## Q7 - Pergunta Livre

_A preencher via `/insight-livre`._